# Callable(可调用对象)

1. 在python中，只要能加`()`执行的东西都成为 **Callable**

2. Typing是Python 是动态类型语言：变量不需要提前声明类型，运行时自动判断。

Callable 的标准写法为：

`Callable[[参数类型1, 参数类型2, ...], 返回值类型]`

- 外层 `Callable[...]`：表示“这是一个可调用类型提示”
- 内层 `[...]`：表示“这个可调用对象接收**哪些参数**”（必须用列表/元组包起来）
- 逗号后面：表示“**执行后返回什么类型**”

|你想标注的函数|对应的 Callable 写法|解释|
|-------------|-------------------|-------|
|def greet(name: str) -> str:|Callable[[str], str]|接收1个字符串，返回1个字符串|
|def add(a: int, b: int) -> int:|Callable[[int, int], int]|接收2个整数，返回1个整数|
|def setup() -> None:|Callable[[], None]|不接收参数，不返回值|
|Any fun()|Callable 或 Callable[..., Any]|只要能加 () 就行，不关心参数和返回值

In [1]:
from typing import Callable

In [7]:
def run_tool(tool: Callable[[str], str]):
    result = tool("hello world")
    return result


In [4]:
run_tool("Im not a func")

TypeError: 'str' object is not callable

In [9]:
def func(text: str) -> str:
    print(f"正在执行我的 func，收到: {text}")
    return f"处理完成: {text.upper()}"

run_tool(func)

正在执行我的 func，收到: hello world


'处理完成: HELLO WORLD'

# \_\_call\_\_

`__call__` 是 Python 的魔法方法`（Magic Method）`。它的作用是：

> 让类的实例对象变得“可调用”，即支持 **`实例(参数)`** 的写法。

In [10]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor
    
    def __call__(self, x):
        return x * self.factor

In [12]:
doubler = Multiplier(2)
doubler(5)

10

Callable 类型提示不仅能接收 def 函数，也能接收实现了 \_\_call\_\_ 的类实例。这在现代框架中极其常见：
- PyTorch 的 nn.Module 前向传播靠的就是 model(x) → 底层走 \_\_call\_\_
- LangChain / ReAct Agent 的“有状态工具”常封装成带 \_\_call\_\_ 的类
- 装饰器工厂、配置化回调等场景都依赖它

In [21]:
def run_tool(tool: Callable[[int], int], value: int) -> int:
    return tool(value)

In [ ]:
run_tool(doubler, 5)        # 输出: 10 (5 * 2)

10

# \_\_func\_\_

\_\_func\_\_ 是 绑定方法（Bound Method）对象的专属属性。它的作用是：

> 拿到方法背后那个“最原始的函数对象”。

在 Python 中，当你通过实例访问方法时（如 `obj.method`），Python 并不会直接返回函数，而是创建一个绑定方法对象，它内部打包了两样东西：
- `__self__`：绑定的实例对象（obj）
- `__func__`：类定义时的原始函数

In [17]:
class Agent:
    def search(self, query) -> str:
        return f"query: {query}"

In [18]:
a = Agent()
print(type(a.search))
print(a.search.__func__)
print(a.search.__self__)

<class 'method'>
<function Agent.search at 0x000001CEC1C9FEB0>


In [20]:
# 通过 __func__ 也能调用（但需手动传self）
print(a.search.__func__(a, "Python"))

query: Python


为什么开源项目爱用 `__func__`？
1. 序列化/持久化：绑定方法带实例状态，不好存库。提取 `__func__` 后可统一按函数处理。

2. 动态注册路由：如 FastAPI / Flask 装饰器内部常用 `func.__func__` 统一提取函数名做映射。

3. 绕过绑定限制：在元编程、AOP、Mock 测试中需要直接操作底层函数。

In [1]:
import os 

path = os.path.abspath()

TypeError: abspath() missing 1 required positional argument: 'path'